<a href="https://colab.research.google.com/github/LAC1012/Portable-Info-Structure/blob/main/I3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [ ]:
url = "https://www.washington.edu/students/crscat/imt.html"

response = requests.get(url)
print(response.status_code)
print(response.text[:100])


200
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">


<html>



In [ ]:
soup = BeautifulSoup(response.text,"html.parser")
print(soup.get_text("\n")[:20000])









INFORMATION MANAGEMENT & TECHNOLOGY













     
Search
 |
    
Directories
 |
    
Reference
    Tools










UW Home
 > 
Discover UW
 > 
Student Guide
 



	UW Bothell Course Descriptions



	UW Tacoma Course Descriptions



	Glossary





THE INFORMATION SCHOOL

THE INFORMATION SCHOOL

INFORMATION MANAGEMENT & TECHNOLOGY



Detailed course offerings (Time Schedule) are available for


Spring Quarter 2026: 
Time Schedule (NetID required)
 | 
Course Offerings


Summer Quarter 2026: 
Time Schedule (NetID required)
 | 
Course Offerings


Autumn Quarter 2026: 
Time Schedule (NetID required)
 | 
Course Offerings




IMT 500 Foundations of Information Management (4)
Examines the role and function of information and information management in individual, organizational, community, and social contexts. Topics include defining information and information management concepts; overview of systems and systems architecture, methods of managing information and information flows wit

In [ ]:
p_tags = soup.find_all("p")

print("how mant p tags:", len(p_tags))

for i, p in enumerate(p_tags[:10]):
    print(f"--- p tag {i} ---")
    print(p.get_text(" ", strip=True))
    print()

how mant p tags: 40
--- p tag 0 ---
Detailed course offerings (Time Schedule) are available for Spring Quarter 2026: Time Schedule (NetID required) | Course Offerings Summer Quarter 2026: Time Schedule (NetID required) | Course Offerings Autumn Quarter 2026: Time Schedule (NetID required) | Course Offerings

--- p tag 1 ---
IMT 500 Foundations of Information Management (4) Examines the role and function of information and information management in individual, organizational, community, and social contexts. Topics include defining information and information management concepts; overview of systems and systems architecture, methods of managing information and information flows within organizations; and internal and external communication in professional settings. View course details in MyPlan: IMT 500

--- p tag 2 ---
IMT 511 Introduction to Programming for Information and Data Science (4) Introduces fundamentals of computer programming as used for data science. Covers foundational skil

In [ ]:
sample = p_tags[1].get_text(" ", strip=True)
print(sample)

IMT 500 Foundations of Information Management (4) Examines the role and function of information and information management in individual, organizational, community, and social contexts. Topics include defining information and information management concepts; overview of systems and systems architecture, methods of managing information and information flows within organizations; and internal and external communication in professional settings. View course details in MyPlan: IMT 500


In [ ]:
cleaned = sample.split("View course details in MyPlan:")[0].strip()
print(cleaned)

IMT 500 Foundations of Information Management (4) Examines the role and function of information and information management in individual, organizational, community, and social contexts. Topics include defining information and information management concepts; overview of systems and systems architecture, methods of managing information and information flows within organizations; and internal and external communication in professional settings.


In [ ]:
import re

match = re.match(r"^(IMT\s+\d+)\s+(.+?)\s+\((\d+)\)\s+(.*)$", cleaned)

if match:
    course_id = match.group(1)
    course_title = match.group(2)
    credits = match.group(3)
    description = match.group(4)

    print("course_id:", course_id)
    print("course_title:", course_title)
    print("credits:", credits)
    print("description:", description)
else:
    print("No match found")

course_id: IMT 500
course_title: Foundations of Information Management
credits: 4
description: Examines the role and function of information and information management in individual, organizational, community, and social contexts. Topics include defining information and information management concepts; overview of systems and systems architecture, methods of managing information and information flows within organizations; and internal and external communication in professional settings.


In [ ]:
b_tags = soup.find_all("b")

titles = []
for b in b_tags:
    text = b.get_text(" ", strip=True)
    if text.startswith("IMT "):
        titles.append(text)

print("how many titles:", len(titles))
for t in titles[:10]:
    print(t)

how many titles: 39
IMT 500 Foundations of Information Management (4)
IMT 511 Introduction to Programming for Information and Data Science (4)
IMT 519 Information Science Study Abroad (1-8, max. 18)
IMT 526 Building and Applying Large Language Models (4)
IMT 530 Organization of Information Resources (4)
IMT 535 Introduction to Information Architecture (5)
IMT 540 Design Methods for Interactive Systems (4)
IMT 541 Enterprise Information Systems Analysis and Design (4)
IMT 542 Portable Information Structures (4)
IMT 543 Relational Database Management Systems (4)


In [ ]:
import re
import pandas as pd

rows = []

for b in b_tags:
    title_text = b.get_text(" ", strip=True)

    if not title_text.startswith("IMT "):
        continue

    match = re.match(r"^(IMT\s+\d+)\s+(.+?)\s+\((.*?)\)$", title_text)
    if not match:
        continue

    course_id = match.group(1)
    course_title = match.group(2)
    credits = match.group(3)

    description = ""

    for p in p_tags:
        p_text = p.get_text(" ", strip=True)
        if title_text in p_text:
            description = p_text.replace(title_text, "").split("View course details in MyPlan:")[0].strip()
            break

    rows.append({
        "course_id": course_id,
        "course_title": course_title,
        "credits": credits,
        "description": description
    })

df = pd.DataFrame(rows)
print(df.shape)
df

(39, 4)


,course_id,course_title,credits,description
0,IMT 500,Foundations of Information Management,4,Examines the role and function of information ...
1,IMT 511,Introduction to Programming for Information an...,4,Introduces fundamentals of computer programmin...
2,IMT 519,Information Science Study Abroad,"1-8, max. 18","International seminar, led by Information Scho..."
3,IMT 526,Building and Applying Large Language Models,4,Introduces the building blocks of large langua...
4,IMT 530,Organization of Information Resources,4,Introduction to issues in organization of info...
5,IMT 535,Introduction to Information Architecture,5,Introduces concepts and methods of front- and ...
6,IMT 540,Design Methods for Interactive Systems,4,Introduction to the theory and practice of use...
7,IMT 541,Enterprise Information Systems Analysis and De...,4,Uses Systems Development Lifecycle (SDLC) as a...
8,IMT 542,Portable Information Structures,4,Introduces the concepts and methods used to an...
9,IMT 543,Relational Database Management Systems,4,"Introduces relational database design, impleme..."


In [ ]:
tag_map = {
    "IMT 526": {
        "career_path_tags": "AI",
        "skill_tags": "llm;foundation models;prompting;applied ai;model evaluation"
    },
    "IMT 530": {
        "career_path_tags": "IA",
        "skill_tags": "metadata;classification;controlled vocabulary;information organization;taxonomy"
    },
    "IMT 535": {
        "career_path_tags": "IA",
        "skill_tags": "information architecture;content modeling;navigation;labeling;taxonomy;seo"
    },
    "IMT 540": {
        "career_path_tags": "UX",
        "skill_tags": "user-centered design;design research;prototyping;interaction design;user needs"
    },
    "IMT 541": {
        "career_path_tags": "PPMC",
        "skill_tags": "systems analysis;enterprise systems;requirements;process design;solution design"
    },
    "IMT 542": {
        "career_path_tags": "IA",
        "skill_tags": "information structures;navigation;content structure;portability;information design"
    },
    "IMT 555": {
        "career_path_tags": "InfoSec",
        "skill_tags": "cybersecurity;risk;security fundamentals;threats;security strategy"
    },
    "IMT 558": {
        "career_path_tags": "InfoSec",
        "skill_tags": "security leadership;governance;risk management;compliance;security program management"
    },
    "IMT 559": {
        "career_path_tags": "InfoSec",
        "skill_tags": "cybersecurity trends;security technologies;technical leadership;risk analysis"
    },
    "IMT 561": {
        "career_path_tags": "UX",
        "skill_tags": "data visualization;visual design;human-centered design;coding;data storytelling"
    },
    "IMT 565": {
        "career_path_tags": "UX",
        "skill_tags": "experience design;ux;service design;customer experience;design strategy"
    },
    "IMT 572": {
        "career_path_tags": "BI",
        "skill_tags": "data science fundamentals;data literacy;analytics;data workflows;applied data"
    },
    "IMT 573": {
        "career_path_tags": "DS",
        "skill_tags": "data science;statistics;modeling;technical foundations;data analysis"
    },
    "IMT 574": {
        "career_path_tags": "DS",
        "skill_tags": "machine learning;predictive modeling;data analysis;model evaluation"
    },
    "IMT 575": {
        "career_path_tags": "DS",
        "skill_tags": "scaling data science;big data;ethics;data systems;applications"
    },
    "IMT 576": {
        "career_path_tags": "BI",
        "skill_tags": "business intelligence;decision making;analytics strategy;dashboards;reporting"
    },
    "IMT 577": {
        "career_path_tags": "BI",
        "skill_tags": "bi systems;etl;data warehousing;dimensional modeling;decision support"
    },
    "IMT 585": {
        "career_path_tags": "PPMC",
        "skill_tags": "consulting;program management;stakeholder management;business problem solving"
    },
    "IMT 587": {
        "career_path_tags": "PPMC",
        "skill_tags": "project management;planning;execution;stakeholder communication;business context"
    }
}

In [ ]:
tag_map.update({
    "IMT 500": {
        "career_path_tags": "General",
        "skill_tags": "information management;systems thinking;professional communication;organizational context"
    },
    "IMT 511": {
        "career_path_tags": "General;DS;AI",
        "skill_tags": "python;programming;data structures;debugging;computational thinking"
    },
    "IMT 543": {
        "career_path_tags": "DS;BI",
        "skill_tags": "sql;database design;relational databases;data management"
    },
    "IMT 547": {
        "career_path_tags": "DS;BI",
        "skill_tags": "social media analysis;data collection;text analysis;applied analytics"
    },
    "IMT 550": {
        "career_path_tags": "General",
        "skill_tags": "ethics;policy;privacy;information governance"
    },
    "IMT 571": {
        "career_path_tags": "DS",
        "skill_tags": "social network analysis;graph analysis;network methods;data interpretation"
    },
    "IMT 579": {
        "career_path_tags": "BI;PPMC",
        "skill_tags": "people analytics;measurement;decision making;organizational data"
    },
    "IMT 580": {
        "career_path_tags": "PPMC",
        "skill_tags": "leadership;strategy;organizational behavior;management"
    }
})

In [ ]:
df["career_path_tags"] = df["course_id"].map(lambda x: tag_map.get(x, {}).get("career_path_tags", ""))
df["skill_tags"] = df["course_id"].map(lambda x: tag_map.get(x, {}).get("skill_tags", ""))

In [ ]:
df[["course_id", "career_path_tags", "skill_tags"]].head(10)

,course_id,career_path_tags,skill_tags
0,IMT 500,General,information management;systems thinking;profes...
1,IMT 511,General;DS;AI,python;programming;data structures;debugging;c...
2,IMT 519,,
3,IMT 526,AI,llm;foundation models;prompting;applied ai;mod...
4,IMT 530,IA,metadata;classification;controlled vocabulary;...
5,IMT 535,IA,information architecture;content modeling;navi...
6,IMT 540,UX,user-centered design;design research;prototypi...
7,IMT 541,PPMC,systems analysis;enterprise systems;requiremen...
8,IMT 542,IA,information structures;navigation;content stru...
9,IMT 543,DS;BI,sql;database design;relational databases;data ...


In [ ]:
import pandas as pd

def recommend_courses(df, keyword):
    keyword = keyword.lower().strip()
    results = []

    for _, row in df.iterrows():
        score = 0

        course_title = str(row.get("course_title", "")).lower()
        description = str(row.get("description", "")).lower()
        career_path_tags = str(row.get("career_path_tags", "")).lower()
        skill_tags = str(row.get("skill_tags", "")).lower()

        if keyword in course_title:
            score += 3
        if keyword in career_path_tags:
            score += 4
        if keyword in skill_tags:
            score += 3
        if keyword in description:
            score += 1

        if score > 0:
            results.append({
                "course_id": row["course_id"],
                "course_title": row["course_title"],
                "credits": row["credits"],
                "career_path_tags": row["career_path_tags"],
                "skill_tags": row["skill_tags"],
                "score": score
            })

    if results:
        return pd.DataFrame(results).sort_values(by=["score", "course_id"], ascending=[False, True])
    else:
        return pd.DataFrame(columns=[
            "course_id", "course_title", "credits",
            "career_path_tags", "skill_tags", "score"
        ])

In [ ]:
keyword = input("Enter a career path or skill keyword: ")
recommend_courses(df, keyword)

Enter a career path or skill keyword: UX


,course_id,course_title,credits,career_path_tags,skill_tags,score
2,IMT 565,Designing Information Experiences,4,UX,experience design;ux;service design;customer e...,7
0,IMT 540,Design Methods for Interactive Systems,4,UX,user-centered design;design research;prototypi...,4
1,IMT 561,Data Visualization: Design and Development,4,UX,data visualization;visual design;human-centere...,4
